# 4. 平均限界効果

この notebook では、**平均限界効果（AME）**を推定します。その際に、推定対象を表す線型汎函数 $m$ を書くだけで二重機械学習を実行できることを確認します。

平均限界効果は、連続処置や連続入力に対して「少しだけ動かしたときにアウトカムが平均的にどれだけ変わるか」を表す量です。  
スライドの一般形では

$$
\theta = \mathbb{E}[m(X,\gamma_0)]
$$

という書き方をしましたが、AME もまさにこの形に入ります。

この notebook で見たいことは、微分型の推定対象も、$m(x,\gamma)$ を自分で書けば 二重機械学習 + リース回帰 = 自動バイアス除去学習で推定できることです。


In [1]:
import numpy as np
import pandas as pd

from genriesz import (
    CallableFunctional,
    grr_functional,
    SquaredGenerator,
    PolynomialBasis,
)

rng = np.random.default_rng(0)


## DGP

ここでは真値が分かりやすいように、三次多項式を含む回帰関数を使います。

$$
Y = 1 + X_0 + 0.3 X_0^3 + 0.5 X_1^2 + \varepsilon
$$

このとき、$X_0$ 方向の偏微分は

$$
\frac{\partial}{\partial x_0}\gamma(x)
= 1 + 0.9 x_0^2
$$

です。したがって、平均限界効果は

$$
\theta_{\mathrm{AME}}
= \mathbb{E}\left[\frac{\partial}{\partial x_0}\gamma(X)\right]
= 1 + 0.9\,\mathbb{E}[X_0^2]
$$

になります。

今回は $X$ を平均 0 の二変量正規分布から生成しており、$X_0$ の分散は既知なので、AME の真値も解析的に計算できます。

ただし、この notebook では微分そのものではなく、あとで**中央差分**

$$
m_h(x,\gamma)=\frac{\gamma(x+h e_k)-\gamma(x-h e_k)}{2h}
$$

を使って実装します。  
そのため、比較対象としては「厳密な AME」と「有限差分で近似したターゲット」の 2 つを表示しています。


In [2]:
n = 1500
mean = np.zeros(2)
cov = np.array([[1.0, 0.4], [0.4, 1.0]])

X = rng.multivariate_normal(mean, cov, size=n)
eps = rng.normal(scale=1.0, size=n)

Y = 1.0 + X[:, 0] + 0.3 * (X[:, 0] ** 3) + 0.5 * (X[:, 1] ** 2) + eps

coordinate = 0
h = 1e-3

true_ame = 1.0 + 0.9 * cov[0, 0]   # = 1.9
true_fd_target = true_ame + 0.3 * h**2  # central difference on the cubic term

print(f"Population AME                    : {true_ame:.6f}")
print(f"Finite-difference target (h={h:g}): {true_fd_target:.6f}")


Population AME                    : 1.900000
Finite-difference target (h=0.001): 1.900000


## 線型汎函数 $m_h$ を定義する

AME は本来、回帰関数の微分を平均した量です。しかし実装では、まず

$$
m_h(x,\gamma)=\frac{\gamma(x+h e_k)-\gamma(x-h e_k)}{2h}
$$

という中央差分作用素を使います。ここで $e_k$ は第 $k$ 成分だけが 1 の単位ベクトルです。

この式は、回帰関数 $\gamma$ に対して線形です。  
したがって、これも一般形

$$
\theta_h = \mathbb{E}[m_h(X,\gamma_0)]
$$

の一例になっています。

コードでは `make_ame_fd_functional` がこの $m_h$ を返します。  $h$ を十分小さく取れば、$\theta_h$ は AME のよい近似になります。


In [3]:
def make_ame_fd_functional(coordinate: int, h: float):
    def m(x_row, gamma):
        x_plus = np.array(x_row, dtype=float, copy=True)
        x_minus = np.array(x_row, dtype=float, copy=True)
        x_plus[coordinate] += h
        x_minus[coordinate] -= h
        return (gamma(x_plus) - gamma(x_minus)) / (2.0 * h)

    return m


## 推定関数を作る



### 1. 基底の選択

DGP に三次項が含まれているので、`PolynomialBasis(degree=3)` を使っています。  
この設定では、真の回帰関数が基底の張る空間にかなり近いため、回帰関数の近似が比較的うまくいくことが期待できます。

### 2. 推定量の種類

`estimators=("ra", "rw", "arw", "tmle")` として、回帰調整型・重み付け型・補正付き型・TMLE 型をまとめて計算します。 

共変量 $Z$ には二次多項式基底を当てています。

### 3. `cross_fit=True` で交差適合の有無を選択する

- `cross_fit=False` なら、同じデータで局外母数を学習し、そのまま使う
- `cross_fit=True` なら、fold ごとに外部サンプルで局外母数を学習し、推定対象の fold には out-of-fold 予測だけを使う

という違いになります。

In [4]:
basis = PolynomialBasis(degree=3, include_bias=True)
gen = SquaredGenerator(C=0.0).as_generator()

m_ame = CallableFunctional(
    make_ame_fd_functional(coordinate=coordinate, h=h),
    name=f"AME via custom m (coord={coordinate}, h={h:g})",
)

def fit_ame_generic(*, cross_fit: bool, folds: int = 3):
    return grr_functional(
        X=X,
        Y=Y,
        m=m_ame,
        basis=basis,
        generator=gen,
        cross_fit=cross_fit,
        folds=folds,
        random_state=0,
        estimators=("ra", "rw", "arw", "tmle"),
        outcome_models="shared",
        outcome_link="identity",
        riesz_penalty="l2",
        riesz_lam=1e-3,
        max_iter=200,
        tol=1e-8,
    )


In [5]:
res_plugin = fit_ame_generic(cross_fit=False)
res_dml = fit_ame_generic(cross_fit=True, folds=3)

print("=== plug-in / no cross-fitting ===")
print(res_plugin.summary_text())
print()
print("=== DML / cross-fitting ===")
print(res_dml.summary_text())


=== plug-in / no cross-fitting ===
AME via custom m (coord=0, h=0.001) estimates (n=1500)
alpha=0.05 | null=0.0

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 1.91724     0.0328437         [ 1.85286,  1.98161]           0
RW                 1.91686      0.132408         [ 1.65735,  2.17638]           0
ARW                1.91761       0.04307          [ 1.8332,  2.00203]           0
TMLE               1.91761     0.0430613         [ 1.83321,  2.00201]           0

=== DML / cross-fitting ===
AME via custom m (coord=0, h=0.001) estimates (n=1500)
alpha=0.05 | null=0.0

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 1.91455     0.0326066         [ 1.85064,  1.97846]           0
RW                 1.99571      0.149467  

## 比較表の見方

表では、plug-in 版と cross-fit 版の推定値を並べています。

今回の真値として直接比較しているのは、厳密な AME そのものではなく、**この notebook で実装した中央差分作用素 $m_h$ に対応する有限差分ターゲット**です。  
これは、コードで実際に推定している対象と評価対象を一致させるためです。

ここで注目してほしい点は 2 つあります。

1. **推定対象を自分で定義しても**、`grr_functional` が標準誤差や信頼区間まで返してくれること
2. `cross_fit=True` によって、一般形の問題でも DML 的な推定に切り替わること

つまり、AME のような微分型の問題も、実装上は「特殊な別問題」ではなく、**線型汎函数 $m$ を指定した一般の debiased learning 問題**として扱えます。


In [6]:
def records_from_result(res, fit_label: str, true_value: float):
    rows = []
    for key, est in res.estimates.items():
        rows.append(
            {
                "fit": fit_label,
                "estimator": key.upper(),
                "estimate": est.estimate,
                "se": est.se,
                "ci_low": est.ci_low,
                "ci_high": est.ci_high,
                "error_vs_true": est.estimate - true_value,
            }
        )
    return rows

comparison_df = pd.DataFrame(
    records_from_result(res_plugin, "no cross-fit", true_fd_target)
    + records_from_result(res_dml, "cross-fit = DML", true_fd_target)
)

comparison_df = comparison_df.sort_values(["fit", "estimator"]).reset_index(drop=True)
comparison_df


,fit,estimator,estimate,se,ci_low,ci_high,error_vs_true
0,cross-fit = DML,ARW,1.927380,0.044711,1.839749,2.015012,0.027380
1,cross-fit = DML,RA,1.914547,0.032607,1.850640,1.978455,0.014547
2,cross-fit = DML,RW,1.995714,0.149467,1.702764,2.288665,0.095714
3,cross-fit = DML,TMLE,1.926805,0.044388,1.839806,2.013804,0.026805
4,no cross-fit,ARW,1.917613,0.043070,1.833197,2.002028,0.017612
5,no cross-fit,RA,1.917237,0.032844,1.852864,1.981609,0.017236
6,no cross-fit,RW,1.916862,0.132408,1.657347,2.176376,0.016861
7,no cross-fit,TMLE,1.917613,0.043061,1.833214,2.002012,0.017613


In [7]:
print("Interpretation:")
print(f"- true AME                          = {true_ame:.6f}")
print(f"- finite-difference target m_h      = {true_fd_target:.6f}")
print("- cross-fitted ARW / TMLE are the generic DML-style estimators here.")


Interpretation:
- true AME                          = 1.900000
- finite-difference target m_h      = 1.900000
- cross-fitted ARW / TMLE are the generic DML-style estimators here.


## この notebook のまとめ

この notebook では、平均限界効果を一般形から実装しました。

- AME は回帰関数の微分を平均した量である。
- その微分作用素を中央差分 $m_h(x,\gamma)$ として書けば、`grr_functional` にそのまま渡せる。
- したがって、built-in の `grr_ame` を使わなくても、一般 API だけで AME 型の推定ができる。
- さらに `cross_fit=True` を指定すると、局外母数推定が out-of-fold になり、自動で DML が走る。

これで、ATE・DiD・AME がいずれも「推定対象を線型汎函数として書き、必要なら交差適合で debias する」という**ひとつの統一的な見方**で実装できることが確認できました。
